In [217]:
import pandas as pd

In [218]:
df = pd.read_csv("Loan_default.csv") # Load the data set in a variable

In [219]:
df.head() # Display the head

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


In [220]:
df.columns

Index(['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore',
       'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm',
       'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus',
       'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner',
       'Default'],
      dtype='str')

In [221]:
df.shape

(255347, 18)

In [222]:
df["Default"].value_counts() # Find the total counts

Default
0    225694
1     29653
Name: count, dtype: int64

In [223]:
df["Default"].value_counts(normalize=True) * 100 # Put them into percentage, for balancing

Default
0    88.387175
1    11.612825
Name: proportion, dtype: float64

In [224]:
df.isnull().sum() # Check if there is any missing values to use KNN

LoanID            0
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64

In [225]:
# Match the numeric features to a list
numeric_features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio"
]

In [226]:
# Match the categorical features to a list
categorical_features = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

In [227]:
# Find the mean values of the numeric features to determine it is okay to test
df.groupby("Default")[numeric_features].mean()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio
Default,,,,,,,,,
0,44.409962,83899.165995,125353.656017,576.232270,60.764721,2.489566,13.176994,36.022544,0.498602
1,36.559539,71844.722659,144515.311469,559.286143,50.235457,2.588338,15.896227,36.051394,0.512467


In [228]:
# Find the mean values of the categorical features to determine it is okay to test
df.groupby("Education")["Default"].mean()

Education
Bachelor's     0.121011
High School    0.128789
Master's       0.108717
PhD            0.105860
Name: Default, dtype: float64

In [229]:
df.groupby("EmploymentType")["Default"].mean()

EmploymentType
Full-time        0.094634
Part-time        0.119652
Self-employed    0.114620
Unemployed       0.135529
Name: Default, dtype: float64

In [230]:
df.groupby("Education")["Default"].mean().sort_values(ascending=False)

Education
High School    0.128789
Bachelor's     0.121011
Master's       0.108717
PhD            0.105860
Name: Default, dtype: float64

In [231]:
df.groupby("EmploymentType")["Default"].mean().sort_values(ascending=False)

EmploymentType
Unemployed       0.135529
Part-time        0.119652
Self-employed    0.114620
Full-time        0.094634
Name: Default, dtype: float64

In [232]:
categorical_features = [
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

for feature in categorical_features:
    print(f"\nDefault rate by {feature}:")
    print(df.groupby(feature)["Default"].mean().sort_values(ascending=False))


Default rate by Education:
Education
High School    0.128789
Bachelor's     0.121011
Master's       0.108717
PhD            0.105860
Name: Default, dtype: float64

Default rate by EmploymentType:
EmploymentType
Unemployed       0.135529
Part-time        0.119652
Self-employed    0.114620
Full-time        0.094634
Name: Default, dtype: float64

Default rate by MaritalStatus:
MaritalStatus
Divorced    0.125328
Single      0.119124
Married     0.103972
Name: Default, dtype: float64

Default rate by HasMortgage:
HasMortgage
No     0.123451
Yes    0.108806
Name: Default, dtype: float64

Default rate by HasDependents:
HasDependents
No     0.127244
Yes    0.105024
Name: Default, dtype: float64

Default rate by LoanPurpose:
LoanPurpose
Business     0.123260
Auto         0.118814
Education    0.118381
Other        0.117885
Home         0.102348
Name: Default, dtype: float64

Default rate by HasCoSigner:
HasCoSigner
No     0.128661
Yes    0.103601
Name: Default, dtype: float64


In [233]:
features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio",
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

target = "Default"

X = df[features]
y = df[target]

In [234]:
X_encoded = pd.get_dummies(X, drop_first=True) # Useful to match reach row to the specific category, dummy encoding

In [235]:
X_encoded.head() # Show the new rows with the correct identified feature

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education_High School,...,EmploymentType_Unemployed,MaritalStatus_Married,MaritalStatus_Single,HasMortgage_Yes,HasDependents_Yes,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other,HasCoSigner_Yes
0,56,85994,50587,520,80,4,15.23,36,0.44,False,...,False,False,False,True,True,False,False,False,True,True
1,69,50432,124440,458,15,1,4.81,60,0.68,False,...,False,True,False,False,False,False,False,False,True,True
2,46,84208,129188,451,26,3,21.17,24,0.31,False,...,True,False,False,True,True,False,False,False,False,False
3,32,31713,44799,743,0,3,7.07,24,0.23,True,...,False,True,False,False,False,True,False,False,False,False
4,60,20437,9139,633,8,4,6.51,48,0.73,False,...,True,False,False,False,True,False,False,False,False,False


In [236]:
X_encoded.dtypes # Check if all types are correct

Age                               int64
Income                            int64
LoanAmount                        int64
CreditScore                       int64
MonthsEmployed                    int64
NumCreditLines                    int64
InterestRate                    float64
LoanTerm                          int64
DTIRatio                        float64
Education_High School              bool
Education_Master's                 bool
Education_PhD                      bool
EmploymentType_Part-time           bool
EmploymentType_Self-employed       bool
EmploymentType_Unemployed          bool
MaritalStatus_Married              bool
MaritalStatus_Single               bool
HasMortgage_Yes                    bool
HasDependents_Yes                  bool
LoanPurpose_Business               bool
LoanPurpose_Education              bool
LoanPurpose_Home                   bool
LoanPurpose_Other                  bool
HasCoSigner_Yes                    bool
dtype: object

In [237]:
print(X_encoded.shape)
print(y.shape)

(255347, 24)
(255347,)


In [238]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size = 0.2, random_state=0, stratify=y) # Setting up configuration, stratify helps keep same class balance 

In [239]:
# Using Logistic Regression
from sklearn.linear_model import LogisticRegression 

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train) # Model did not fully converge

KeyboardInterrupt: 

In [ ]:
# Columns has many different ranges, so the use of scaling puts them in a comparable scale
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train) 

In [ ]:
# Take out the model predictions, predict_proba gives probability of two classes
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1] 

In [ ]:
# Get the current model predictions and compare them with each score, threshold is 0.5 with sigmoid function
# Accuracy = (TP + TN) / Total and shows how accurate the model training is
# Precision = TP / (TP + FP) && Recall = TP / (TP + FN)
# ROC-AUC measures how well the model separates defaulters from non-defaulters using probabilities.
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "ROC-AUC": roc_auc
}

plt.figure(figsize=(8, 5))
plt.bar(metrics.keys(), metrics.values())
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Logistic Regression Model Performance")
plt.show()

In [ ]:
# Model does not identify all the defaults correctly, find a different threshold to train
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

threshold = 0.25

y_pred_custom = (y_proba >= threshold).astype(int)

print("Precision:", precision_score(y_test, y_pred_custom))
print("Recall:", recall_score(y_test, y_pred_custom))
print("F1:", f1_score(y_test, y_pred_custom))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

In [ ]:
# Test for other thresholds to see what is better, lowering thresholds
for threshold in [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    y_pred_custom = (y_proba >= threshold).astype(int)
    print("Threshold:", threshold)
    print("Precision:", precision_score(y_test, y_pred_custom))
    print("Recall:", recall_score(y_test, y_pred_custom))
    print("F1:", f1_score(y_test, y_pred_custom))
    print(confusion_matrix(y_test, y_pred_custom))
    print()

In [ ]:
# Chose the model based on F1 score
# F1 = 2 × (Precision × Recall) / (Precision + Recall)
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

precisions = []
recalls = []
f1_scores = []

for threshold in thresholds:
    y_pred_custom = (y_proba >= threshold).astype(int)

    precisions.append(precision_score(y_test, y_pred_custom))
    recalls.append(recall_score(y_test, y_pred_custom))
    f1_scores.append(f1_score(y_test, y_pred_custom))

plt.figure(figsize=(8, 5))
plt.plot(thresholds, precisions, marker="o", label="Precision")
plt.plot(thresholds, recalls, marker="o", label="Recall")
plt.plot(thresholds, f1_scores, marker="o", label="F1 Score")

plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall, and F1 Score by Threshold")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Display Confusion Matrix that shows TN FP FN TP
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

threshold = 0.20
y_pred_020 = (y_proba >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred_020)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-default", "Default"]
)

disp.plot()
plt.title("Confusion Matrix at Threshold 0.20")
plt.show()

## Threshold Tuning

Because the dataset is imbalanced, the default classification threshold of 0.50 produced very low recall. I tested multiple thresholds from 0.10 to 0.50 and compared precision, recall, and F1 score.

Lower thresholds increased recall but reduced precision, while higher thresholds increased precision but missed more defaulters. The threshold of 0.20 gave the best F1 score, making it the strongest balanced choice for this MVP model.

At threshold 0.20, the model correctly identified 2,567 defaulters, compared to only 220 defaulters at the default threshold of 0.50.

In [ ]:
# Use Random Forest that can catch features like low income + high loan amount + high interest rate
# Number of estimators is the amount of decision trees 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_model.fit(X_train, y_train)

rf_proba = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
# Compare with all thresholds
thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

rf_precisions = []
rf_recalls = []
rf_f1_scores = []

for threshold in thresholds:
    rf_pred = (rf_proba >= threshold).astype(int)

    precision = precision_score(y_test, rf_pred)
    recall = recall_score(y_test, rf_pred)
    f1 = f1_score(y_test, rf_pred)

    rf_precisions.append(precision)
    rf_recalls.append(recall)
    rf_f1_scores.append(f1)

    print("Threshold:", threshold)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("ROC-AUC:", roc_auc_score(y_test, rf_proba))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, rf_pred))
    print()

In [ ]:
# Visualize to see the best threshold
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.plot(thresholds, rf_precisions, marker="o", label="Precision")
plt.plot(thresholds, rf_recalls, marker="o", label="Recall")
plt.plot(thresholds, rf_f1_scores, marker="o", label="F1 Score")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Random Forest: Precision, Recall, and F1 by Threshold")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()

## Random Forest Results

I trained a Random Forest model to compare against the Logistic Regression baseline. Random Forest can capture nonlinear relationships and feature interactions, but in this experiment it did not outperform Logistic Regression.

The best Random Forest threshold was 0.20, with an F1 score of 0.342 and ROC-AUC of 0.731. Logistic Regression performed better with an F1 score of 0.355 and ROC-AUC of 0.750 at the same threshold.

Because Logistic Regression had better recall, F1 score, and ROC-AUC, it remains the stronger model for this MVP.

In [ ]:
# Compare between Logistic Regression and Random Forest
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Best Threshold": [0.20, 0.20],
    "Precision": [0.301, 0.300],
    "Recall": [0.433, 0.398],
    "F1 Score": [0.355, 0.342],
    "ROC-AUC": [0.750, 0.731]
})

comparison

In [ ]:
import matplotlib.pyplot as plt

comparison.set_index("Model")[["Precision", "Recall", "F1 Score", "ROC-AUC"]].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Performance Comparison")
plt.xticks(rotation=0)
plt.legend()
plt.show()

In [ ]:
# Use XGBoost as a new machine learning algorithim

from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# Test with all thresholds

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

for threshold in thresholds:
    xgb_pred = (xgb_proba >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", precision_score(y_test, xgb_pred))
    print("Recall:", recall_score(y_test, xgb_pred))
    print("F1:", f1_score(y_test, xgb_pred))
    print("ROC-AUC:", roc_auc_score(y_test, xgb_proba))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, xgb_pred))
    print()

In [ ]:
# Visualize data to see if XGBoost is better than Logistic Regression

import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]

xgb_precisions = []
xgb_recalls = []
xgb_f1_scores = []

for threshold in thresholds:
    xgb_pred = (xgb_proba >= threshold).astype(int)

    xgb_precisions.append(precision_score(y_test, xgb_pred))
    xgb_recalls.append(recall_score(y_test, xgb_pred))
    xgb_f1_scores.append(f1_score(y_test, xgb_pred))

plt.figure(figsize=(8, 5))

plt.plot(thresholds, xgb_precisions, marker="o", label="Precision")
plt.plot(thresholds, xgb_recalls, marker="o", label="Recall")
plt.plot(thresholds, xgb_f1_scores, marker="o", label="F1 Score")

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("XGBoost: Precision, Recall, and F1 by Threshold")
plt.ylim(0, 1)
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

threshold = 0.15
xgb_pred_015 = (xgb_proba >= threshold).astype(int)

cm = confusion_matrix(y_test, xgb_pred_015)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-default", "Default"]
)

disp.plot()
plt.title("XGBoost Confusion Matrix at Threshold 0.15")
plt.show()

## XGBoost Results

I trained an XGBoost model as a stronger gradient-boosted tree model for loan default prediction. XGBoost performed slightly better than both Logistic Regression and Random Forest.

The best XGBoost threshold was 0.15, with an F1 score of 0.358 and ROC-AUC of 0.753. This model had lower precision than Logistic Regression, but it had much higher recall, meaning it caught more actual defaulters.

For this MVP, XGBoost at threshold 0.15 is the selected model because it provides the best balance overall and catches the most risky borrowers.

In [ ]:
# Test the whole dataset using the selected XGBoost model

import pandas as pd

features = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio",
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner"
]

X_all = df[features]
y_all = df["Default"]

X_all_encoded = pd.get_dummies(X_all, drop_first=True)

X_all_encoded = X_all_encoded.reindex(columns=X_train.columns, fill_value=0)

all_proba = xgb_model.predict_proba(X_all_encoded)[:, 1]

def risk_decision(probability):
    if probability < 0.15:
        return "Low Risk"
    elif probability < 0.40:
        return "Medium Risk - Manual Review"
    else:
        return "High Risk"

results = df.copy()
results["Predicted_Default_Probability"] = all_proba
results["Risk_Decision"] = results["Predicted_Default_Probability"].apply(risk_decision)

results_view = results[
    [
        "Age",
        "Income",
        "LoanAmount",
        "CreditScore",
        "InterestRate",
        "DTIRatio",
        "Default",
        "Predicted_Default_Probability",
        "Risk_Decision"
    ]
]

results_view.head(20)

In [ ]:
results["Risk_Decision"].value_counts()

In [ ]:
results["Risk_Decision"].value_counts(normalize=True) * 100

In [ ]:
results.groupby("Risk_Decision")["Default"].mean().sort_values(ascending=False)

## Risk Category Validation

After selecting the XGBoost model, I applied the model to the full dataset and converted predicted default probabilities into three risk categories: Low Risk, Medium Risk - Manual Review, and High Risk.

The risk categories showed a clear separation in actual default rates:

- Low Risk: 6.57% default rate
- Medium Risk: 24.09% default rate
- High Risk: 56.40% default rate

This confirms that the model’s risk scores are meaningful. Applicants labeled High Risk defaulted far more often than applicants labeled Low Risk, while Medium Risk applicants fell between the two groups.

Because of this, the model is useful as an MVP risk-screening tool rather than a fully automated approval system.

In [ ]:
import matplotlib.pyplot as plt

risk_default_rates = results.groupby("Risk_Decision")["Default"].mean() * 100

risk_default_rates = risk_default_rates.reindex([
    "Low Risk",
    "Medium Risk - Manual Review",
    "High Risk"
])

plt.figure(figsize=(8, 5))
plt.bar(risk_default_rates.index, risk_default_rates.values)
plt.ylabel("Actual Default Rate (%)")
plt.title("Actual Default Rate by Predicted Risk Category")
plt.xticks(rotation=20)
plt.show()

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score

xgb_tuned = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_tuned.fit(X_train, y_train)

xgb_tuned_proba = xgb_tuned.predict_proba(X_test)[:, 1]

threshold = 0.15
xgb_tuned_pred = (xgb_tuned_proba >= threshold).astype(int)

print("F1:", f1_score(y_test, xgb_tuned_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_tuned_proba))

## Different Selected Model

After comparing Logistic Regression, Decision Tree, Random Forest, XGBoost, LightGBM, and a tuned XGBoost model, I selected the tuned XGBoost model with a threshold of 0.15.

The tuned XGBoost model achieved the best F1 score of 0.361 and ROC-AUC of 0.753. This made it the strongest model for the MVP risk-screening system.

In [ ]:
# Check the best threshold starting from 0.5.
from sklearn.metrics import precision_score, recall_score, f1_score

best_threshold = None
best_f1 = 0

for threshold in [i / 100 for i in range(5, 100)]:
    pred = (xgb_tuned_proba >= threshold).astype(int)
    f1 = f1_score(y_test, pred)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best F1:", best_f1)

In [ ]:
# 0.17 is the best based on the F1 score,

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

final_threshold = 0.17

final_pred = (xgb_tuned_proba >= final_threshold).astype(int)

print("Precision:", precision_score(y_test, final_pred))
print("Recall:", recall_score(y_test, final_pred))
print("F1:", f1_score(y_test, final_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_tuned_proba))
print("Confusion Matrix:")
print(confusion_matrix(y_test, final_pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

threshold = 0.17
xgb_pred_015 = (xgb_proba >= threshold).astype(int)

cm = confusion_matrix(y_test, xgb_pred_015)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-default", "Default"]
)

disp.plot()
plt.title("XGBoost Confusion Matrix at Threshold 0.17")
plt.show()

## Selected Model

The final selected model is the tuned XGBoost model with a decision threshold of 0.17.

After threshold tuning, the best F1 score improved to 0.363. This threshold gave the best balance between precision and recall for the MVP risk-screening system.

In [ ]:
# Provide an explanity first for each feature
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_tuned.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.head(15)

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importances - Tuned XGBoost")
plt.gca().invert_yaxis()
plt.show()

### Explainability

To make the tuned XGBoost model more interpretable, I analyzed feature importance. Feature importance shows which variables had the greatest influence on the model's predictions.

The most important feature was Age, followed by InterestRate, Income, MonthsEmployed, and LoanAmount. These features are reasonable for a loan default prediction problem because they relate to borrower stability, loan affordability, and repayment risk.

Employment-related features such as MonthsEmployed, EmploymentType_Self-employed, and EmploymentType_Unemployed also appeared in the top features, showing that the model considers employment stability when estimating default risk.

Co-signer and dependent status also influenced predictions, which makes sense because they may affect repayment responsibility and financial obligations.

This explainability step helps make the model less of a black box. In a production lending system, more advanced tools such as SHAP or LIME would be needed to explain individual predictions and support compliance, audits, and fair lending review.

In [ ]:
# Bias and Fairness Checks

import pandas as pd

# Rebuild results using the original unencoded test rows
# This gives us readable columns like Education, EmploymentType, MaritalStatus, etc.
fairness_df = df.loc[X_test.index].copy()

fairness_df["Actual_Default"] = y_test.values
fairness_df["Predicted_Default_Probability"] = xgb_tuned_proba
fairness_df["Risk_Decision"] = fairness_df["Predicted_Default_Probability"].apply(risk_decision)

# Create age groups
fairness_df["Age_Group"] = pd.cut(
    fairness_df["Age"],
    bins=[18, 30, 45, 60, 100],
    labels=["18-30", "31-45", "46-60", "60+"],
    include_lowest=True
)

# Create income groups
fairness_df["Income_Group"] = pd.cut(
    fairness_df["Income"],
    bins=[0, 40000, 80000, 120000, 200000],
    labels=["Low Income", "Middle Income", "Upper Middle Income", "High Income"],
    include_lowest=True
)

print("Average Predicted Default Probability by Age Group")
print(fairness_df.groupby("Age_Group", observed=False)["Predicted_Default_Probability"].mean())
print("\n")

print("Actual Default Rate by Age Group")
print(fairness_df.groupby("Age_Group", observed=False)["Actual_Default"].mean())
print("\n")

print("Risk Decision Distribution by Age Group (%)")
print(
    pd.crosstab(
        fairness_df["Age_Group"],
        fairness_df["Risk_Decision"],
        normalize="index"
    ) * 100
)
print("\n")

print("Average Predicted Default Probability by Income Group")
print(fairness_df.groupby("Income_Group", observed=False)["Predicted_Default_Probability"].mean())
print("\n")

print("Actual Default Rate by Income Group")
print(fairness_df.groupby("Income_Group", observed=False)["Actual_Default"].mean())
print("\n")

print("Risk Decision Distribution by Income Group (%)")
print(
    pd.crosstab(
        fairness_df["Income_Group"],
        fairness_df["Risk_Decision"],
        normalize="index"
    ) * 100
)
print("\n")

print("Average Predicted Default Probability by Education")
print(fairness_df.groupby("Education")["Predicted_Default_Probability"].mean())
print("\n")

print("Average Predicted Default Probability by Employment Type")
print(fairness_df.groupby("EmploymentType")["Predicted_Default_Probability"].mean())
print("\n")

print("Average Predicted Default Probability by Marital Status")
print(fairness_df.groupby("MaritalStatus")["Predicted_Default_Probability"].mean())
print("\n")

print("Risk Decision Distribution by Employment Type (%)")
print(
    pd.crosstab(
        fairness_df["EmploymentType"],
        fairness_df["Risk_Decision"],
        normalize="index"
    ) * 100
)

## Bias and Fairness Checks

Bias and fairness checks are important in lending because credit models can affect financial access. A model should not unfairly discriminate against protected or sensitive groups.

This dataset does not include gender, race, or region, so fairness could not be fully evaluated across all protected groups. However, fairness-style checks were performed using the available fields, including age groups, income groups, education, employment type, and marital status.

The model assigned higher predicted default probabilities to younger borrowers and lower-income borrowers. These groups also had higher actual default rates in the test data, which suggests that the model is capturing patterns present in the dataset rather than assigning risk randomly.

For age groups, the 18-30 group had the highest predicted default probability and the highest actual default rate. Older age groups had lower predicted risk and lower actual default rates.

For income groups, low-income borrowers had the highest predicted default probability and the highest actual default rate. Higher-income groups had lower predicted risk and lower actual default rates.

Education groups had relatively similar predicted default probabilities, with no extreme differences across categories. Employment type showed slightly higher predicted risk for unemployed borrowers, which is consistent with the idea that employment stability can affect repayment ability.

Overall, these checks suggest that the model's risk scores generally align with actual default patterns in the dataset. However, before using this model in a real lending system, additional fairness testing would be required using more complete demographic data, regulatory review, and human oversight.

In [ ]:
# Attempt feature engineering

df_fe = df.copy()

df_fe["LoanIncomeRatio"] = df_fe["LoanAmount"] / df_fe["Income"]
df_fe["InterestBurden"] = df_fe["LoanAmount"] * df_fe["InterestRate"]
df_fe["EmploymentStabilityRatio"] = df_fe["MonthsEmployed"] / df_fe["Age"]
df_fe["CreditIncomeRatio"] = df_fe["CreditScore"] / df_fe["Income"]
df_fe["LoanAmount_DTI"] = df_fe["LoanAmount"] * df_fe["DTIRatio"]

In [ ]:
features_fe = [
    "Age",
    "Income",
    "LoanAmount",
    "CreditScore",
    "MonthsEmployed",
    "NumCreditLines",
    "InterestRate",
    "LoanTerm",
    "DTIRatio",
    "Education",
    "EmploymentType",
    "MaritalStatus",
    "HasMortgage",
    "HasDependents",
    "LoanPurpose",
    "HasCoSigner",
    "LoanIncomeRatio",
    "InterestBurden",
    "EmploymentStabilityRatio",
    "CreditIncomeRatio",
    "LoanAmount_DTI"
]

X_fe = df_fe[features_fe]
y_fe = df_fe["Default"]

X_fe_encoded = pd.get_dummies(X_fe, drop_first=True)

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe_encoded,
    y_fe,
    test_size=0.2,
    random_state=42,
    stratify=y_fe
)

In [ ]:
xgb_fe = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_fe.fit(X_train_fe, y_train_fe)

xgb_fe_proba = xgb_fe.predict_proba(X_test_fe)[:, 1]

In [ ]:
best_threshold_fe = None
best_f1_fe = 0

for threshold in [i / 100 for i in range(5, 51)]:
    pred = (xgb_fe_proba >= threshold).astype(int)
    f1 = f1_score(y_test_fe, pred)

    if f1 > best_f1_fe:
        best_f1_fe = f1
        best_threshold_fe = threshold

print("Best threshold:", best_threshold_fe)
print("Best F1:", best_f1_fe)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test_fe, final_pred_fe)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-Default", "Default"]
)

disp.plot()
plt.title("Confusion Matrix - Feature-Engineered Tuned XGBoost")
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

precision = precision_score(y_test_fe, final_pred_fe)
recall = recall_score(y_test_fe, final_pred_fe)
f1 = f1_score(y_test_fe, final_pred_fe)
roc_auc = roc_auc_score(y_test_fe, xgb_fe_proba)

metrics = {
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc
}

plt.figure(figsize=(8, 5))
plt.bar(metrics.keys(), metrics.values())
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Model Performance - Feature-Engineered Tuned XGBoost")
plt.show()

## Updated Business Risk Categorization

After adding feature-engineered variables, I reran the business risk categorization using the feature-engineered XGBoost model.

The updated risk groups continued to separate borrowers meaningfully:

- Low Risk borrowers had an actual default rate of about 7.05%.
- Medium Risk borrowers had an actual default rate of about 26.05%.
- High Risk borrowers had an actual default rate of about 53.75%.

This shows that the feature-engineered model not only improved F1-score and ROC-AUC, but also maintained strong business usefulness by separating applicants into clear risk tiers.

In [240]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(xgb_fe, "models/xgb_feature_engineered_loan_default_model.pkl")
joblib.dump(list(X_train_fe.columns), "models/model_columns.pkl")
joblib.dump(best_threshold_fe, "models/best_threshold.pkl")

['models/best_threshold.pkl']